In [40]:
import pandas as pd

In [41]:
df=pd.read_csv('dataset/train.csv')

In [42]:
import re
import numpy as np

In [43]:
def extract_keys(content):
    if pd.isna(content):
        return {}
    # Regex to match "Key: Value" patterns
    pattern = re.compile(r'([\w\s\d]+?):\s(.+)')
    matches = pattern.findall(content)
    # Convert to dict
    d = {}
    for k, v in matches:
        d[k.strip()] = v.strip()
    return d

# Apply extraction
expanded = df['catalog_content'].apply(extract_keys).apply(pd.Series)

# Combine with sample_id
result = pd.concat([df['sample_id'], expanded], axis=1)

# Optional: normalize columns (ensure all expected columns exist)
expected_cols = ['sample_id',
                 'Item Name',
                 'Bullet Point 1', 'Bullet Point 2', 'Bullet Point 3', 'Bullet Point 4', 'Bullet Point 5',
                 'Product Description',
                 'Value', 'Unit']
result = result.reindex(columns=expected_cols)

# Save to CSV
result.to_csv("test_normalised.csv", index=False)

print("CSV saved as test_normalised.csv")

CSV saved as test_normalised.csv


In [44]:
result.shape

(75000, 10)

In [45]:
result.columns

Index(['sample_id', 'Item Name', 'Bullet Point 1', 'Bullet Point 2',
       'Bullet Point 3', 'Bullet Point 4', 'Bullet Point 5',
       'Product Description', 'Value', 'Unit'],
      dtype='object')

In [46]:
result['Unit'].unique()

array(['Fl Oz', 'Ounce', 'Count', 'ounce', 'None', 'Fluid Ounce', 'count',
       'oz', 'Ounces', 'pound', 'fl oz', 'gram', 'grams', 'COUNT',
       'FL Oz', 'lb', 'Each', 'Liters', 'gramm', 'ct', 'Pound', 'Oz',
       'OZ', 'millilitre', 'Jar', 'ounces', 'Fl. Oz', 'bottle', 'Bottle',
       'Gram', 'Can', 'Tea Bags', 'Fl oz', 'each', '24', 'Pack', 'Piece',
       'fluid ounces', 'gr', 'milliliter', 'mililitro', 'CT', 'FL OZ',
       'pack', 'pounds', 'kg', 'Bag', 'in', 'fl. oz.', 'K-Cups',
       'fluid ounce', 'sq ft', '-', 'ml', 'Packs', 'box', '8', 'Fl Ounce',
       'Pouch', 'Bucket', 'LB', 'per Box', 'Per Package',
       'fluid ounce(s)', 'units', 'packs', 'BOX/12', '1', 'Fluid Ounces',
       'product_weight', 'Sq Ft', 'per Carton', 'Foot', 'Grams(gm)',
       'Box', 'unità', 'Paper Cupcake Liners', 'capsule', 'bottles',
       'bag', nan, '---', 'Fl.oz', 'Pounds', 'Ziplock bags',
       'Fluid ounce', 'ltr', 'PACK', 'can', 'Carton', 'Tea bags',
       '7,2 oz'], dtype=object)

In [ ]:
import numpy as np
import pandas as pd
import re

# Assume your DataFrame is called result

# --- Step 1: Map units to standard categories ---
unit_mapping = {
    # Weight
    'oz': 'oz', 'Ounce': 'oz', 'ounce': 'oz', 'Oz': 'oz', 'OZ': 'oz', 'ounces': 'oz', 'Ounces': 'oz',
    'lb': 'lb', 'lbs': 'lb', 'Pound': 'lb', 'pound': 'lb', 'Pounds': 'lb',
    'g': 'g', 'gram': 'g', 'grams': 'g', 'Gram': 'g', 'Grams': 'g',
    'kg': 'kg',
    # Volume
    'fl_oz': 'fl_oz', 'Fl Oz': 'fl_oz', 'Fl. Oz': 'fl_oz', 'fl oz': 'fl_oz', 'FL OZ': 'fl_oz',
    'fluid ounce': 'fl_oz', 'Fluid Ounce': 'fl_oz', 'fluid ounces': 'fl_oz', 'Fluid Ounces': 'fl_oz',
    'fluid ounce(s)': 'fl_oz', 'fl.oz': 'fl_oz', 'Fl Ounce': 'fl_oz', 'Fl. OZ': 'fl_oz',
    'ml': 'ml', 'milliliter': 'ml', 'millilitre': 'ml', 'mililitro': 'ml',
    'l': 'l', 'Liter': 'l', 'Liters': 'l', 'Ltr': 'l', 'gal': 'gal', 'Gallon': 'gal', 'gallons': 'gal',
    # Count-based
    'count': 'count', 'Count': 'count', 'COUNT': 'count', 'each': 'count', 'Each': 'count',
    'ct': 'count', 'CT': 'count', 'EA': 'count', 'ea': 'count', 'unit': 'count',
    'Piece': 'count', 'Pieces': 'count', 'per Package': 'count', 'Per Package': 'count',
    '(Pack of 1)': 'count', 'tea bags': 'count', 'Tea Bags': 'count', 'stück': 'count',
    # Packs / containers (approximate as weight)
    'pack': 'pack', 'Pack': 'pack', 'packs': 'pack', 'Packs': 'pack', 'PACK': 'pack',
    'Packet': 'pack', 'BAG': 'pack', 'Bag': 'pack', 'bag': 'pack',
    'Box': 'pack', 'BOX': 'pack', 'Container': 'pack', 'Jar': 'pack', 'jar': 'pack',
    'JARS': 'pack', 'Tin': 'pack', 'Pouch': 'pack', 'SACHET': 'pack', 'KIT': 'pack',
    'K-Cups': 'pack', 'Paper Cupcake Liners': 'pack', 'Bottle': 'pack', 'bottle': 'pack',
    # fallback
    'none': 'none', 'other': 'none'
}

# --- Step 2: Conversion factors ---
conversion = {
    # Weight
    'oz': 28.3495,
    'lb': 453.592,
    'g': 1.0,
    'kg': 1000.0,
    # Volume
    'fl_oz': 29.5735,
    'ml': 1.0,
    'l': 1000.0,
    'gal': 3785.41,
    # Count-based
    'count': 1.0,
    # Packs approximated
    'pack': 200.0
}

# --- Step 3: Unit type for NN ---
unit_type_mapping = {
    'oz': 'weight', 'lb': 'weight', 'g': 'weight', 'kg': 'weight',
    'fl_oz': 'volume', 'ml': 'volume', 'l': 'volume', 'gal': 'volume',
    'count': 'count', 'pack': 'count'
}

# --- Step 4: Parse numeric value ---
def parse_value(v):
    try:
        v = str(v).replace(',', '.')
        match = re.findall(r"[\d\.]+", v)
        return float(match[0]) if match else 1.0
    except:
        return 1.0

result['Value'] = result['Value'].apply(parse_value)
result['unit_std'] = result['Unit'].map(unit_mapping).fillna('none')

# --- Step 5: Calculate quantity ---
result['quantity'] = result.apply(
    lambda row: row['Value'] * conversion.get(row['unit_std'], 1.0),
    axis=1
)

# --- Step 6: Add unit_type for NN ---
result['unit_type'] = result['unit_std'].map(unit_type_mapping).fillna('count')

# --- Step 7: Drop original Value, Unit, unit_std ---
result = result.drop(columns=['Value', 'Unit', 'unit_std'])

# --- Step 8: Save ---
result.to_csv('dataset/train_normalised.csv', index=False)
print("✅ Dataset ready with quantity and unit_type, other columns preserved.")


✅ Dataset ready with quantity and unit_type, other columns preserved.


In [48]:
result.shape

(75000, 10)

In [49]:
result.columns

Index(['sample_id', 'Item Name', 'Bullet Point 1', 'Bullet Point 2',
       'Bullet Point 3', 'Bullet Point 4', 'Bullet Point 5',
       'Product Description', 'quantity', 'unit_type'],
      dtype='object')

In [50]:
result.head()

,sample_id,Item Name,Bullet Point 1,Bullet Point 2,Bullet Point 3,Bullet Point 4,Bullet Point 5,Product Description,quantity,unit_type
0,33127,"La Victoria Green Taco Sauce Mild, 12 Ounce (P...",NaN,NaN,NaN,NaN,NaN,NaN,2129.292000,volume
1,198967,"Salerno Cookies, The Original Butter Cookies, ...",Original Butter Cookies: Classic butter cookie...,Variety Pack: Includes 4 boxes with 32 cookies...,Occasion Perfect: Delicious cookies for birthd...,Shareable Treats: Fun to give and enjoy with f...,Salerno Brand: Trusted brand of delicious butt...,NaN,907.184000,weight
2,261251,"Bear Creek Hearty Soup Bowl, Creamy Chicken wi...",Loaded with hearty long grain wild rice and ve...,Full of hearty goodness,Single serve bowls,Easy to prepare mix,0 grams trans fat,NaN,323.184300,weight
3,55858,Judee’s Blue Cheese Powder 11.25 oz - Gluten-F...,"Add to your favorite appetizers, dips & spread...","Sprinkle over french fries, fried chicken, mas...",Made in a dedicated gluten-free facility and s...,"Ingredients: Blue Cheese (Milk, Salt, Cultures...","Since 2009, Judee’s has been dedicated to prov...",Judees Powdered Blue Cheese cheddar cheese pow...,318.931875,weight
4,292686,"kedem Sherry Cooking Wine, 12.7 Ounce - 12 per...",NaN,NaN,NaN,NaN,NaN,NaN,12.000000,count
